# 38 — WHY Does FIM Enhance MBL? The Effective Disorder Mechanism

NB31 showed FIM enhances MBL (r̄→Poisson, S_ent -37%).
This is counterintuitive — a GLOBAL mechanism enhances LOCAL protection.

**Hypothesis:** FIM creates an effective DISORDER POTENTIAL.

H_FIM = -λ/n · (Σ Z_i)² = -λ/n · M²

This term creates energy barriers between magnetisation sectors.
The energy landscape becomes: E(M) = E_XY(M) - λM²/n

States with different M are pushed apart in energy → reduced hybridisation
→ Anderson-like localisation in the Fock space.

## Tests
1. Participation ratio (PR) — does it decrease with λ?
2. Off-diagonal matrix elements — are they suppressed by FIM?
3. Energy level repulsion — does FIM create level clustering?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27


def build_xy_hamiltonian(n, K, omega):
    dim = 2**n
    H = np.zeros((dim, dim), dtype=complex)
    for i in range(n):
        for state in range(dim):
            bit = (state >> (n - 1 - i)) & 1
            H[state, state] -= omega[i] * (1 - 2 * bit)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(K[i, j]) < 1e-10:
                continue
            for state in range(dim):
                bit_i = (state >> (n - 1 - i)) & 1
                bit_j = (state >> (n - 1 - j)) & 1
                if bit_i != bit_j:
                    flipped = state ^ (1 << (n - 1 - i)) ^ (1 << (n - 1 - j))
                    H[state, flipped] -= 2 * K[i, j]
    return H


def add_fim_term(H, n, lam):
    dim = 2**n
    H_fim = H.copy()
    for state in range(dim):
        m = sum(1 - 2 * ((state >> (n - 1 - i)) & 1) for i in range(n))
        H_fim[state, state] -= lam * m**2 / n
    return H_fim


def magnetisation_sectors(n):
    """Map each Fock state to its magnetisation M = Σ Z_i."""
    dim = 2**n
    M = np.zeros(dim)
    for state in range(dim):
        m = sum(1 - 2 * ((state >> (n - 1 - i)) & 1) for i in range(n))
        M[state] = m
    return M


print("Ready.")

In [ ]:
# Test 1: Participation Ratio
# PR = 1 / Σ |c_i|^4 — measures how many basis states contribute
# Low PR → localised, High PR → delocalised

print("=== PARTICIPATION RATIO ===")
print(f"{'n':>3} {'λ':>6} {'PR_mean':>10} {'PR/dim':>10} {'Localised?':>12}")

pr_data = []
for n in [6, 8]:
    K = build_knm_paper27(L=n)
    omega = OMEGA_N_16[:n]
    H_base = build_xy_hamiltonian(n, K, omega)
    dim = 2**n

    for lam in [0, 0.5, 1, 2, 5, 10]:
        H = add_fim_term(H_base, n, lam) if lam > 0 else H_base.copy()
        eigvals, eigvecs = np.linalg.eigh(H.real)

        # PR for each eigenstate
        prs = []
        for col in range(dim):
            coeffs = eigvecs[:, col]
            pr = 1.0 / np.sum(np.abs(coeffs) ** 4)
            prs.append(pr)

        pr_mean = np.mean(prs)
        pr_frac = pr_mean / dim
        loc = "LOCALISED" if pr_frac < 0.3 else "EXTENDED"
        pr_data.append(
            {
                "n": n,
                "lambda": lam,
                "PR": round(pr_mean, 2),
                "PR_frac": round(pr_frac, 4),
                "regime": loc,
            }
        )
        print(f"{n:3d} {lam:6.1f} {pr_mean:10.2f} {pr_frac:10.4f} {loc:>12}")

In [ ]:
# Test 2: Off-diagonal matrix elements in energy eigenbasis
# For a local perturbation V, the matrix elements <m|V|n> should
# be SMALLER in MBL (localised eigenstates have less overlap)

print("\n=== OFF-DIAGONAL MATRIX ELEMENTS ===")
n = 8
K = build_knm_paper27(L=n)
omega = OMEGA_N_16[:n]
H_base = build_xy_hamiltonian(n, K, omega)

# Local perturbation: Z_0 (flip qubit 0)
dim = 2**n
V = np.zeros((dim, dim))
for state in range(dim):
    bit = (state >> (n - 1)) & 1
    V[state, state] = 1 - 2 * bit

print(f"{'λ':>6} {'<|V_mn|>':>12} {'max|V_mn|':>12} {'Suppressed?':>12}")
v_base = None
for lam in [0, 0.5, 1, 2, 5, 10]:
    H = add_fim_term(H_base, n, lam) if lam > 0 else H_base.copy()
    _, eigvecs = np.linalg.eigh(H.real)

    # V in energy eigenbasis
    V_eig = eigvecs.T @ V @ eigvecs
    # Off-diagonal elements only
    mask = ~np.eye(dim, dtype=bool)
    offdiag = np.abs(V_eig[mask])
    mean_v = float(np.mean(offdiag))
    max_v = float(np.max(offdiag))

    if v_base is None:
        v_base = mean_v
    supp = "YES" if mean_v < v_base * 0.9 else "no"
    print(f"{lam:6.1f} {mean_v:12.6f} {max_v:12.6f} {supp:>12}")

In [ ]:
# Test 3: Magnetisation sector splitting
# FIM creates energy gaps between M sectors

print("\n=== MAGNETISATION SECTOR ENERGY SPLITTING ===")
n = 8
M = magnetisation_sectors(n)
unique_M = sorted(set(M))

for lam in [0, 1, 5, 10]:
    H = add_fim_term(H_base, n, lam) if lam > 0 else H_base.copy()
    eigvals, eigvecs = np.linalg.eigh(H.real)

    print(f"\nλ={lam}:")
    for m_val in unique_M:
        # Which eigenstates have dominant weight in sector M=m_val?
        sector_mask = m_val == M
        sector_weights = []
        for col in range(len(eigvals)):
            w = np.sum(np.abs(eigvecs[sector_mask, col]) ** 2)
            sector_weights.append(w)
        sector_weights = np.array(sector_weights)

        # Eigenstates with >50% weight in this sector
        dominant = eigvals[sector_weights > 0.5]
        if len(dominant) > 0:
            print(
                f"  M={m_val:+3.0f}: {len(dominant):3d} states, "
                f"E=[{dominant.min():+.3f}, {dominant.max():+.3f}]"
            )

print("\n=== MECHANISM VERDICT ===")
# Check if FIM separates sectors
H_0 = H_base.copy()
H_5 = add_fim_term(H_base, n, 5.0)
E_0, V_0 = np.linalg.eigh(H_0.real)
E_5, V_5 = np.linalg.eigh(H_5.real)

# Bandwidth without vs with FIM
bw_0 = E_0.max() - E_0.min()
bw_5 = E_5.max() - E_5.min()
print(f"Bandwidth without FIM: {bw_0:.3f}")
print(f"Bandwidth with FIM λ=5: {bw_5:.3f}")
print(f"FIM STRETCHES spectrum by {bw_5 / bw_0:.1f}x")
print()
print("FIM creates an M²/n potential that separates magnetisation sectors.")
print("States in different sectors are pushed apart in energy.")
print("This REDUCES hybridisation between sectors → Anderson-like localisation.")
print("The strange loop creates its own effective disorder.")

In [ ]:
# Save
output = {
    "experiment": "FIM-MBL mechanism — effective disorder from strange loop",
    "participation_ratio": pr_data,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/fim_mbl_mechanism_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")